In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import imageio.v2 as iio
import nibabel as nib
from skimage import measure
from tqdm import tqdm
import cv2

# ---------- helpers básicos ----------
def load_img_rgb(path):
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

def load_mask_ids(path):
    m = iio.imread(path)
    if m.ndim == 3:
        m = m[..., 0]
    return m.astype(np.int32)

def _sanitize_u8(arr):
    if arr is None:
        return None
    if arr.dtype == np.void or arr.dtype == object:
        try:
            arr = np.array(arr.tolist())
        except Exception:
            arr = np.asarray(arr)
    if not np.issubdtype(arr.dtype, np.integer):
        arr = np.asarray(arr, dtype=np.float32)
        arr = np.clip(arr, 0, 255)
    return arr.astype(np.uint8, copy=False)

def _resize_u8_cv(img_u8, out_wh, interp=cv2.INTER_AREA):
    img_u8 = _sanitize_u8(img_u8)
    ow, oh = out_wh
    return cv2.resize(img_u8, (ow, oh), interpolation=interp).astype(np.uint8, copy=False)

def _resize_bin_cv(bin_u8, out_wh):
    return _resize_u8_cv(bin_u8, out_wh, interp=cv2.INTER_NEAREST)

def _crop_with_padding(img, bbox, pad=6):
    y0, x0, y1, x1 = bbox
    H, W = img.shape[:2]
    y0 = max(y0 - pad, 0); x0 = max(x0 - pad, 0)
    y1 = min(y1 + pad, H); x1 = min(x1 + pad, W)
    return img[y0:y1, x0:x1]

def _candidates(a2d):
    return [
        ("id",           a2d),
        ("rot90",        np.rot90(a2d, 1)),
        ("rot180",       np.rot90(a2d, 2)),
        ("rot270",       np.rot90(a2d, 3)),
        ("transpose",    a2d.T),
        ("flipud",       np.flipud(a2d)),
        ("fliplr",       np.fliplr(a2d)),
        ("rot90+fliplr", np.fliplr(np.rot90(a2d, 1))),
        ("rot90+flipud", np.flipud(np.rot90(a2d, 1))),
    ]

# ---------- NIfTI robusto + alinhamento (patch NumPy 2.0 + dtype estruturado) ----------
def _best_infected_map_from_nii_cv(nii_path, target_shape, cell_union,
                                   force_slice=None, force_transform=None,
                                   _debug_first_fails=5):
    H, W = target_shape
    status = "ok"; dbg_reason = None
    arr3d = None
    try:
        nif = nib.load(str(nii_path))
        try: arr3d = np.asanyarray(nif.dataobj)
        except Exception as e: dbg_reason = f"asanyarray EXC {e}"; arr3d = None
        if arr3d is None or getattr(arr3d, "dtype", None) in (None, np.void, object):
            try: arr3d = nif.get_fdata(dtype=np.float32)
            except Exception as e: dbg_reason = f"get_fdata EXC {e}"; arr3d = None
        if arr3d is not None and (arr3d.dtype == np.void or arr3d.dtype == object):
            try: arr3d = np.asarray(arr3d.tolist(), dtype=np.float32)
            except Exception as e: dbg_reason = f"tolist EXC {e}"; arr3d = None
        if arr3d is None:
            try: arr3d = np.asarray(nif.dataobj, dtype=np.float32)
            except Exception as e: dbg_reason = f"array(dataobj) EXC {e}"; arr3d = None
    except Exception as e:
        dbg_reason = f"nib.load EXC {e}"; arr3d = None

    if arr3d is None or arr3d.size == 0:
        status = "failed"
        if _best_infected_map_from_nii_cv.__dict__.get("_fails", 0) < _debug_first_fails:
            print(f"[WARN] NIfTI FAIL {nii_path.name} :: {dbg_reason}")
            _best_infected_map_from_nii_cv.__dict__["_fails"] = _best_infected_map_from_nii_cv.__dict__.get("_fails", 0) + 1
        return np.zeros((H, W), dtype=np.uint8), -1, "NA", 0, status

    if hasattr(arr3d, "dtype") and arr3d.dtype.fields is not None:
        arr_u8 = arr3d.view(np.uint8).reshape(arr3d.shape + (arr3d.dtype.itemsize,))
        arr3d  = arr_u8.max(axis=-1).astype(np.float32, copy=False)
    elif arr3d.dtype != np.float32:
        arr3d = np.asarray(arr3d, dtype=np.float32)

    if arr3d.ndim >= 3 and arr3d.shape[-1] == 1:
        arr3d = np.squeeze(arr3d, axis=-1)

    if np.max(arr3d) <= 0:
        status = "empty"
        return np.zeros((H, W), dtype=np.uint8), 0, "id", 0, status

    Z = arr3d.shape[-1] if arr3d.ndim == 3 else 1
    best = {"score": -1, "k": None, "name": None, "res": None}
    k_range = [force_slice] if force_slice is not None else range(Z)
    def _bin(x): return (x > 0).astype(np.uint8)

    for k in k_range:
        sl = arr3d[..., k] if arr3d.ndim == 3 else arr3d
        sl = _bin(sl)
        if sl.shape != (H, W):
            sl = _resize_bin_cv(sl, (W, H))
        cand_list = _candidates(sl)
        if force_transform is not None:
            cand_list = [(n, a) for (n, a) in cand_list if n == force_transform]
            if not cand_list:
                raise ValueError(f"Transformação '{force_transform}' não reconhecida.")
        for name, cand in cand_list:
            if cand.shape != (H, W):
                cand = _resize_bin_cv(cand, (W, H))
            cand_bin = _bin(cand)
            inter = np.count_nonzero((cand_bin == 1) & (cell_union == 1))
            if inter > best["score"]:
                best.update(score=int(inter), k=k, name=name, res=cand_bin)

    if best["res"] is None:
        status = "failed"
        if _best_infected_map_from_nii_cv.__dict__.get("_fails", 0) < _debug_first_fails:
            print(f"[WARN] NIfTI FAIL (no candidate) {nii_path.name}")
            _best_infected_map_from_nii_cv.__dict__["_fails"] = _best_infected_map_from_nii_cv.__dict__.get("_fails", 0) + 1
        return np.zeros((H, W), dtype=np.uint8), -1, "NA", 0, status

    return best["res"].astype(np.uint8), int(best["k"]), str(best["name"]), int(best["score"]), status

# ---------- EXPORT estrito (NIfTI manda) ----------
def export_cells_strict_nii(
    img_dir, mask_raw_dir, nii_dir, out_dir,
    split_name="train",
    resize_to=(224, 224),
    dilate_px=2,                # dilatação do NIfTI para tolerar desalinhamento (0=desliga)
    min_cell_area=0,            # ignora células muito pequenas (área em px; 0=desliga)
    force_slice=None,           # define para pular busca (ex.: 0)
    force_transform=None,       # define para pular busca (ex.: "transpose")
    skip_existing=True,
):
    img_dir      = Path(img_dir)
    mask_raw_dir = Path(mask_raw_dir)
    nii_dir      = Path(nii_dir)
    out_dir      = Path(out_dir)

    out_pos = out_dir / split_name / "infected"
    out_neg = out_dir / split_name / "uninfected"
    out_pos.mkdir(parents=True, exist_ok=True)
    out_neg.mkdir(parents=True, exist_ok=True)

    imgs = sorted(img_dir.glob("*.bmp"))
    print(f"{len(imgs)} imagens encontradas em {img_dir}")

    records, decisions = [], []
    missing_mask, missing_nii, empty_masks = [], [], []
    count_ok = count_empty = count_failed = 0

    for img_path in tqdm(imgs):
        stem = img_path.stem

        cand_masks = list(mask_raw_dir.glob(f"{stem}*raw*.png")) or list(mask_raw_dir.glob(f"{stem}*.png"))
        if not cand_masks:
            missing_mask.append(stem); continue
        mask_path = cand_masks[0]

        nii_path = nii_dir / f"{stem}.nii.gz"
        if not nii_path.exists():
            missing_nii.append(stem); continue

        img  = _sanitize_u8(load_img_rgb(img_path))
        mask = load_mask_ids(mask_path)
        if np.max(mask) == 0:
            empty_masks.append(stem); continue

        H, W = mask.shape
        cell_union = (mask > 0).astype(np.uint8)

        infected_map, k, transf, score, nii_status = _best_infected_map_from_nii_cv(
            nii_path, (H, W), cell_union,
            force_slice=force_slice, force_transform=force_transform
        )
        if nii_status == "ok":       count_ok += 1
        elif nii_status == "empty":  count_empty += 1
        else:                        count_failed += 1

        # dilatação opcional para tolerar pequenos deslocamentos
        if dilate_px and dilate_px > 0:
            ksz = 2*dilate_px + 1
            infected_map = cv2.dilate(infected_map, np.ones((ksz, ksz), np.uint8))

        decisions.append({
            "image": stem,
            "slice": int(k),
            "transform": str(transf),
            "score": int(score),
            "nii_status": nii_status
        })

        for r in measure.regionprops(mask):
            rid = r.label
            if min_cell_area > 0 and r.area < min_cell_area:
                continue

            region = (mask == rid)
            inter_px = int(np.count_nonzero(infected_map & region))

            # === rótulo ESTRITO (só NIfTI) ===
            label = 1 if inter_px > 0 else 0

            y0, x0, y1, x1 = r.bbox
            crop = _crop_with_padding(img, (y0, x0, y1, x1), pad=6)
            if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
                continue

            try:
                crop = _resize_u8_cv(crop, resize_to)
            except Exception as e:
                print(f"[WARN] resize falhou em {stem} cell {rid}: dtype={crop.dtype}, shape={crop.shape} | {e}")
                continue

            out_path = (out_pos if label == 1 else out_neg) / f"{stem}_cell{rid:04d}.png"
            if not (skip_existing and out_path.exists()):
                iio.imwrite(out_path, crop)

            records.append({
                "split": split_name,
                "image": stem,
                "cell_id": int(rid),
                "bbox": (int(y0), int(x0), int(y1), int(x1)),
                "label": int(label),
                "path": str(out_path),
                "best_slice": int(k),
                "best_transform": str(transf),
                "align_score": int(score),
                "nii_status": nii_status,
                "inter_px": inter_px,
                "area": int(r.area),
            })

    out_dir.mkdir(parents=True, exist_ok=True)
    df  = pd.DataFrame(records)
    dec = pd.DataFrame(decisions)
    (out_dir / f"cells_{split_name}.csv").write_text(df.to_csv(index=False))
    (out_dir / f"alignment_{split_name}.csv").write_text(dec.to_csv(index=False))

    print("Resumo classes:", df['label'].value_counts(dropna=False).to_dict() if len(df) else "sem registros")
    if missing_mask: print(f"Sem máscara para {len(missing_mask)} imagens (ex.: {missing_mask[:3]})")
    if missing_nii:  print(f"Sem NIfTI para {len(missing_nii)} imagens (ex.: {missing_nii[:3]})")
    if empty_masks:  print(f"Máscaras vazias: {len(empty_masks)} (ex.: {empty_masks[:3]})")
    print(f"NIfTI status → ok: {count_ok} | empty(sem infecção): {count_empty} | failed(corrompido): {count_failed}")

    return df, dec


In [ ]:
export_root   = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_annotated_v3_1"

img_dir_train  = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\train"
img_dir_val    = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\val"

mask_dir_train = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_masks\train_masks\raw"
mask_dir_val   = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_masks\val_masks\raw"

nii_dir_train  = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\annot_dir_cintia\train_segmentadas"
nii_dir_val    = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\annot_dir_cintia\val_segmentadas"

# Busca automática do melhor slice/transform (padrão). 
# force_slice=0, force_transform="transpose"

df_train, dec_train = export_cells_strict_nii(
    img_dir=img_dir_train,
    mask_raw_dir=mask_dir_train,
    nii_dir=nii_dir_train,
    out_dir=export_root,
    split_name="train",
    resize_to=(224,224),
    dilate_px=2,           # ↑ tolere desalinhamento leve (ajuste se precisar) # testes [2,4]
    min_cell_area=0,       # ou, por ex., 120 para filtrar células muito pequenas # testes [0,10]
    # force_slice=None,
    # force_transform=None,
)

df_val, dec_val = export_cells_strict_nii(
    img_dir=img_dir_val,
    mask_raw_dir=mask_dir_val,
    nii_dir=nii_dir_val,
    out_dir=export_root,
    split_name="val",
    resize_to=(224,224),
    dilate_px=2,
    min_cell_area=0,
)


221 imagens encontradas em C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\train


100%|██████████| 221/221 [01:03<00:00,  3.46it/s]


Resumo classes: {0: 3242, 1: 723}
NIfTI status → ok: 215 | empty(sem infecção): 6 | failed(corrompido): 0
221 imagens encontradas em C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\val


100%|██████████| 221/221 [01:07<00:00,  3.26it/s]

Resumo classes: {1: 3535, 0: 276}
NIfTI status → ok: 217 | empty(sem infecção): 4 | failed(corrompido): 0


In [5]:
# diagramna visual do fluxo